In [32]:
import pandas as pd #importamos pandas para el manejo de datos
import yfinance as yf 
import numpy as np
from matplotlib import pyplot as pt

In [33]:
from yfinance import ticker


inicio = '2001-12-01' #fecha en la que empezamos a descargar datos
fin = '2026-03-01' #fecga en la que dejamos de descargar datos

activo = 'AAPL'

datos_activo = yf.download(activo, start=inicio, end=fin, interval='1d') #Descargamos los datos de cierre del activo que queramos analizar
datos_activo = datos_activo.droplevel(level=1, axis=1) #Eliminamos la columna de precio ajustado, ya que no la vamos a usar

[*********************100%***********************]  1 of 1 completed


In [34]:
datos_activo.head()

Price,Close,High,Low,Open,Volume
Date,,,,,
2001-12-03,0.315361,0.318806,0.308619,0.315510,181165600
2001-12-04,0.335586,0.337983,0.310417,0.315361,380419200
2001-12-05,0.355961,0.360005,0.332140,0.334987,568579200
2001-12-06,0.341279,0.352066,0.331690,0.351766,338934400
2001-12-07,0.337683,0.340230,0.329593,0.336484,203515200


In [36]:
datos_activo['return'] = datos_activo['Close'].pct_change() #Calculamos el retorno del activo  
datos_activo['target'] = datos_activo['return'].shift(-1) #Calculamos el target, que es el retorno del siguiente periodo
datos_activo['range'] = (datos_activo['High'] - datos_activo['Low']) / datos_activo['Close']  # Rango normalizado
datos_activo['volume_change'] = datos_activo['Volume'].pct_change()  # Cambio en volumen

# 4. Medias móviles (para dar contexto al LSTM)
datos_activo['close_ma5'] = datos_activo['Close'].rolling(5).mean()
datos_activo['close_ma20'] = datos_activo['Close'].rolling(20).mean()

# 5. Volatilidad
datos_activo['return_vol5'] = datos_activo['return'].rolling(5).std()
datos_activo['return_vol20'] = datos_activo['return'].rolling(20).std()

datos_activo['lag_1'] = datos_activo["return"].shift(1) #Calculamos el lag de 1 mes del precio del activo y lo guardamos en una nueva columna
datos_activo['lag_2'] = datos_activo["return"].shift(2) #Calculamos el lag de 2 meses del precio del activo y lo guardamos en una nueva columna
datos_activo['lag_3'] = datos_activo["return"].shift(3) #Calculamos el lag de 3 meses del precio del activo y lo guardamos en una nueva columna
datos_activo['lag_6'] = datos_activo["return"].shift(6) #Calculamos el lag de 6 meses del precio del activo y lo guardamos en una nueva columna
datos_activo['lag_12'] = datos_activo["return"].shift(12) #Calculamos el lag de 12 meses del precio del activo y lo guardamos en una nueva columna



In [37]:
datos_activo.reset_index() #Reseteamos el index para que la fecha deje de ser el index y pase a ser una columna más
datos_activo.dropna(inplace=True) #Eliminamos las filas con valores nulos, que son las primeras filas que no tienen retorno ni target   
datos_activo.reset_index() #Reseteamos el index para que la fecha deje de ser el index y pase a ser una columna más
datos_activo.head()

Price,Close,High,Low,Open,Volume,return,target,range,volume_change,close_ma5,close_ma20,return_vol5,return_vol20,lag_1,lag_2,lag_3,lag_6,lag_12
Date,,,,,,,,,,,,,,,,,,
2002-01-02,0.349069,0.349069,0.328994,0.330342,529496800,0.063927,0.012017,0.057509,2.842993,0.333159,0.326709,0.031891,0.033359,-0.023631,0.016313,0.026989,0.015965,-0.022801
2002-01-03,0.353264,0.355811,0.341129,0.344574,612007200,0.012017,0.004666,0.041561,0.155828,0.339421,0.327593,0.031447,0.030456,0.063927,-0.023631,0.016313,0.017144,-0.029048
2002-01-04,0.354912,0.358807,0.344425,0.349668,409976000,0.004666,-0.033347,0.040524,-0.330112,0.344275,0.327541,0.031635,0.027280,0.012017,0.063927,-0.023631,0.006085,0.011280
2002-01-07,0.343077,0.359556,0.340829,0.355361,444584000,-0.033347,-0.012665,0.054584,0.084415,0.345683,0.327630,0.038117,0.026699,0.004666,0.012017,0.063927,0.026989,0.018916
2002-01-08,0.338732,0.345324,0.336484,0.340829,450038400,-0.012665,-0.042459,0.026095,0.012269,0.347811,0.327683,0.036352,0.026750,-0.033347,0.004666,0.012017,0.016313,0.029031


In [38]:
datos_activo.to_csv('datos_activo.csv', index=False) #Guardamos el dataframe en un archivo csv para poder usarlo después en el modelo LSTM